In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
# 改了 src/ 里的 .py，notebook 不用重启内核就能用上新代码。
# 魔法命令（% 开头）那一行后面不能写 # 注释，会报 UsageError，所以说明写在这里。
%load_ext autoreload
%autoreload 2

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_mysql import database as mysql
from signal_insight import config, geo, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT   # 配色常量

assert os.environ.get("SIGNAL_MYSQL_PASSWORD"), \
    "没读到 MySQL 密码：关掉 Jupyter，在 PowerShell 里先运行 . .\\.env.ps1，再在同一个窗口运行 jupyter lab"
assert mysql.schema_exists(config.DB_SCHEMA), \
    f"MySQL 里没有库 {config.DB_SCHEMA}：回到 D01-1 第 6 步，用 tools/import_sqlite.py 把 signal.db 迁进来"

viz.apply_theme()                          # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)      # 库名从包里来：notebook 在哪个文件夹打开都一样
DAY = config.ANALYSIS_DAY                  # 分析日期（口径常量）
NB = "D04_这份数据是谁"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"   # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

In [ ]:
from signal_mysql import database as mysql
# 今天要用的两组多边形：读库在这里（调用方），还原交给 geo
DISTRICTS = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_district_boundary", con), "district")
AOI = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", con), "name")
print(f"区县 {len(DISTRICTS)} 个，商圈 {len(AOI)} 个")

In [ ]:
from signal_mysql import database as mysql
# 【跟丢了才运行】D2、D3 没做完的同学：把下面一行的 False 改成 True 再运行，从检查点库复制标准表
CATCH_UP = False
if CATCH_UP:
    with mysql.connect(config.CHECKPOINT_SCHEMA) as ck:
        for t in ["dwd_signal", "dim_cell", "dq_check_result"]:
            mysql.write_frame(mysql.read_frame(f"SELECT * FROM {t}", ck), t, con, if_exists="replace", index=False)
            print("已恢复", t)

In [ ]:
from signal_mysql import database as mysql
# 这一格在做什么：把「清洗后的信令」和「扇区维度表」拼成一张带位置的宽表 sig，
# 今天所有的分析都从它出发。
dwd = mysql.read_frame("SELECT imsi, ts, lac_cell FROM dwd_signal", con, parse_dates=["ts"])
dim = mysql.read_frame("SELECT * FROM dim_cell", con)
# 这里用 inner 而不是 left：D3 已经确认关联率 99.998%，关联不上的 8 条记录没有坐标，
# 留着只会在后面每个空间统计里变成空值。**先验证关联率、再决定用 inner**，顺序不能反。
sig = dwd.merge(dim, on="lac_cell", how="inner")
n_users = sig["imsi"].nunique()
print(f"带位置的信令 {len(sig):,} 条，用户 {n_users:,} 个")

In [ ]:
from signal_mysql import database as mysql
# 这一格在做什么：算每个区县当天有多少比例的样本用户到过。
# ⚠️ 一个人一天可以到过好几个区，所以这些占比**加起来会超过 100%**，不是构成关系。
# 副标题里必须写这句话，否则会当成饼图去理解。
reach = (sig.groupby("district")["imsi"].nunique() / n_users).rename("visitor_share")
reach = reach.sort_values().reset_index()
mysql.write_frame(reach, "ads_district_reach", con, if_exists="replace", index=False)
reach.tail(5)

In [ ]:
from signal_mysql import database as mysql
# ── 出图 D4_01：地图 + 条形图并排 ──────────────────────────────────────
# 为什么是这两种图：地图回答"在空间上集中在哪"，条形图回答"具体差多少"。
#   单给地图，大家分不清 97% 和 85% 的色差；
#   单给条形图，大家不知道这几个区是不是连在一起的。
#   两张并排，一眼就能看出"集中在中心城区，而且和平区几乎是全部"。
# 规范落在哪几行：
#   · BLUES 单色渐变 —— 连续数值不用彩虹色阶（彩虹色阶的颜色顺序不对应数值大小）
#   · viz.maps.set_map_aspect(a1) —— 地图长宽比校正，经纬度不能直接当平面坐标
#   · a2 的条形按大小排序、只给和平区上主色 —— 灰色打底，一种颜色强调一件事
# 标题里的数字：top = 占比最高的那一行（r 已按占比升序排好，所以取 iloc[-1]）。
from matplotlib.colors import LinearSegmentedColormap, Normalize
r = mysql.read_frame("SELECT * FROM ads_district_reach", con)
top = r.iloc[-1]
reach = r.set_index("district")["visitor_share"].sort_values()
BLUES = LinearSegmentedColormap.from_list("blues", ["#EAF2F9", MAIN_MID, MAIN, MAIN_DARK])   # 单色渐变，不用彩虹色阶
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), layout="constrained", width_ratios=[1.25, 1])
norm = Normalize(0, 1)
bx0, bx1, by0, by1 = viz.maps.draw_districts(a1, DISTRICTS, values=reach.to_dict(), cmap=BLUES, norm=norm, edge="white", lw=0.8)
a1.set_xlim(bx0 - 0.05, bx1 + 0.05); a1.set_ylim(by0 - 0.05, by1 + 0.05)
viz.maps.set_map_aspect(a1)          # 长宽比校正 + 去掉刻度、网格、边框
viz.maps.scale_bar(a1, 20); viz.maps.north_arrow(a1)
hp = np.vstack(DISTRICTS["和平区"])                 # 和平区所有顶点，用来定标注的位置
a1.annotate(f"和平区 {reach['和平区']:.0%}", xy=(hp[:, 0].mean(), hp[:, 1].mean()),
            xytext=(hp[:, 0].mean() + 0.35, hp[:, 1].mean() - 0.62), fontsize=9.5, color=INK, fontweight="bold",
            arrowprops=dict(arrowstyle="-", color=INK, lw=0.8))
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=BLUES), ax=a1, shrink=0.55, pad=0.01, location="left")
cb.outline.set_visible(False); cb.ax.tick_params(labelsize=8.5)
cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
bars = a2.barh(reach.index, reach.values, color=[MAIN if d == "和平区" else GRAY for d in reach.index], height=0.66)
a2.bar_label(bars, labels=[f"{v:.0%}" if v >= 0.01 else "<1%" for v in reach.values], padding=3, fontsize=9, color=INK)
a2.set_xlim(0, 1.12); a2.set_xticks([]); a2.grid(False); a2.tick_params(axis="y", labelsize=9, labelcolor=INK)
title = f"{top['visitor_share']:.0%} 的用户当天到过和平区，这不是一份全市随机样本"
viz.finish(fig, title,
           "10-03 到过各区县的用户占比；一个人可以到过多个区，合计超过 100%；左图色深 = 占比高；口径：dwd_signal × dim_cell",
           SOURCE, "D4_01_各区到访用户占比.png")

In [ ]:
# 对账
# ① 封装卡 4.1：dim_cell 是 D03 用它 notebook 里那份函数算的（D03 一个字没改），
#    包里的 geo 必须把每个站点的区县、商圈原样算回来
site = dim.drop_duplicates("site_id")
geo_district = pd.Series(geo.point_in_polygons(site["lng"], site["lat"], DISTRICTS), index=site.index).fillna("市外")
geo_aoi = pd.Series(geo.point_in_polygons(site["lng"], site["lat"], AOI), index=site.index)
assert (geo_district == site["district"]).all(), "geo 落区县和 D03 的 dim_cell 不一致：对照文档检查 src/signal_insight/geo.py"
assert (geo_aoi.fillna("") == site["aoi"].fillna("")).all(), "geo 落商圈和 D03 的 dim_cell 不一致：对照文档检查 src/signal_insight/geo.py"
assert len(site) == 14_106 and (geo_district == "市外").sum() == 4_808
print(f"✅ 对账通过：geo 复现 D03 的 dim_cell，{len(site):,} 个站点的区县、商圈逐个一致（市外 {(geo_district == '市外').sum():,} 个）")
# ② 需求卡 4.1
assert top["district"] == "和平区" and top["visitor_share"] > 0.97
print(f"✅ 对账通过：{top['visitor_share']:.1%} 的用户到过和平区")

In [ ]:
from signal_mysql import database as mysql
site_users = (sig.groupby(["site_id", "lng", "lat"])["imsi"].nunique()
              .rename("n_users").reset_index()
              .sort_values("n_users", ascending=False, ignore_index=True))
site_users["用户占比"] = site_users["n_users"] / n_users
site_users.head(5)

In [ ]:
from signal_mysql import database as mysql
# 为什么取前 5 个站点加权平均，而不是直接用第一名站点的坐标：
# 一座基站覆盖几百米，第一名站点的坐标只是那座铁塔的位置，不是人群的中心。
# 用前 5 名按用户数加权，得到的点更接近真正的客流重心，也不会因为换一天数据就跳到隔壁街区。
# ⚠ 这个坐标算出来一次就**写死进 dim_hotspot（D11 起写进配置文件）**，不随日期重算——
# 否则每天的"西塔"都是一个略微不同的地方，跨日对比就失去意义了。
top5 = site_users.head(5)
hot_lng = np.average(top5["lng"], weights=top5["n_users"])       # 热点中心：前 5 个站点按用户数加权平均
hot_lat = np.average(top5["lat"], weights=top5["n_users"])
poi = mysql.read_frame("SELECT * FROM ods_poi_purpose", con)
poi["距热点(米)"] = geo.haversine(poi["lat"], poi["lng"], hot_lat, hot_lng).round()
poi.nsmallest(6, "距热点(米)")[["name", "type", "距热点(米)"]]

In [ ]:
from signal_mysql import database as mysql
dim_hotspot = pd.DataFrame([{"name": "西塔", "lng": hot_lng, "lat": hot_lat, "radius_m": 500,
                             "note": "连接用户数前 5 的站点按用户数加权的中心；附近为韩百商场、鑫韩城一带"}])
mysql.write_frame(dim_hotspot, "dim_hotspot", con, if_exists="replace", index=False)
dim_hotspot

In [ ]:
# ── 出图 D4_02：气泡地图 ────────────────────────────────────────────────
# 为什么用气泡（面积 ∝ 用户数），不用散点热力：站点只有几十个在视窗里，
# 点数少的时候气泡能直接读出量级差异；点多到上千个时才改用六边形热力（D7_01 就是那种）。
# 规范落在哪几行：
#   · s = top20["用户数"] / 4 —— **面积**与数值成正比（matplotlib 的 s 就是面积）。
#     要是写成半径正比，视觉上会把差距夸大成平方倍
#   · ax.set_aspect(1 / cos(hot_lat)) —— 地图长宽比校正
#   · 画了比例尺（km 那几行）—— 地图没有刻度轴时，必须给读者一个距离参照
# 标题里的数字：site_users.loc[0, "用户占比"]，即排第一的站点连了多少比例的用户。
x0, x1, y0, y1 = hot_lng - 0.035, hot_lng + 0.035, hot_lat - 0.022, hot_lat + 0.022
view = site_users[site_users["lng"].between(x0, x1) & site_users["lat"].between(y0, y1)]
top20 = site_users.head(20)
fig, ax = plt.subplots(figsize=(10, 7.2))
for name, rings in DISTRICTS.items():
    for ring in rings:
        ax.plot(ring[:, 0], ring[:, 1], color=FAINT, lw=1)
ax.scatter(view["lng"], view["lat"], s=4, color=GRAY, lw=0)                 # 所有站点
ax.scatter(top20["lng"], top20["lat"], s=top20["n_users"] / 4, color=MAIN, alpha=0.55, lw=0)   # 前 20，面积 ∝ 用户数
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(hot_lng + 500 / (111_320 * np.cos(np.radians(hot_lat))) * np.cos(theta),
        hot_lat + 500 / 110_570 * np.sin(theta), color=INK, lw=1, ls=(0, (4, 3)))           # 500 米圈
ax.annotate("西塔：韩百商场、鑫韩城一带\n虚线圈半径 500 米", (hot_lng, hot_lat + 500 / 110_570),
            xytext=(0, 10), textcoords="offset points", ha="center", va="bottom", fontsize=10, color=INK)
km = 1 / (111.32 * np.cos(np.radians(hot_lat)))                                     # 1 公里对应的经度
ax.plot([x0 + 0.003, x0 + 0.003 + km], [y0 + 0.002] * 2, color=MUTED, lw=2)
ax.text(x0 + 0.003 + km / 2, y0 + 0.0025, "1 公里", ha="center", va="bottom", fontsize=9, color=MUTED)
ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
ax.set_aspect(1 / np.cos(np.radians(hot_lat)))
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
for sp in ax.spines.values():
    sp.set_visible(False)
title = f"连接人数最多的站点集中在西塔，排第一的站点就有 {site_users.loc[0, '用户占比']:.0%} 的用户"
viz.finish(fig, title,
           "蓝色圆 = 连接用户数前 20 的站点，面积与用户数成正比；灰点 = 其他站点；细线 = 区界；口径：dwd_signal × dim_cell",
           SOURCE, "D4_02_热点地图.png")

In [ ]:
# 对账
assert site_users.loc[0, "n_users"] > 2_200 and poi.nsmallest(1, "距热点(米)")["name"].iloc[0].startswith("韩百")
print(f"✅ 对账通过：第一名站点 {site_users.loc[0, 'n_users']:,} 人；最近的地点是韩百商场")

In [ ]:
from signal_mysql import database as mysql
# 这一格在做什么：算"每个人当天最近到过西塔多远"，再看不同半径下有多少人算"到过"。
# 讲台提示：这是 D8"六种到访人数"的预演——**半径一换，人数就变**。
# 这里先让学生看到 300/500/1000/2000/5000 米五个数，D8 再把时长那一维加进来。
sig["到西塔(米)"] = geo.haversine(sig["lat"], sig["lng"], hot_lat, hot_lng)
user_min = sig.groupby("imsi")["到西塔(米)"].min()                      # 每个人当天离西塔最近的距离
radius = pd.DataFrame({"radius_m": [300, 500, 1000, 2000, 5000]})
radius["reached_share"] = [(user_min <= r).mean() for r in radius["radius_m"]]
mysql.write_frame(radius, "ads_hotspot_radius", con, if_exists="replace", index=False)
radius

In [ ]:
# ── 出图 D4_03：ECDF（累计分布）──────────────────────────────────────────
# 为什么用 ECDF 而不是直方图：读者真正想问的是"多少人在 500 米以内"——
# 这是个**累计**问题。直方图要靠眼睛把左边几根柱加起来，ECDF 直接读纵轴就是答案。
# 分组数（bin 宽度）也不会影响 ECDF 的形状，直方图会。
# 规范落在哪几行：
#   · set_xscale("log") —— 距离跨了 100 米到 30 公里三个数量级，线性轴会把近处全挤成一条
#   · set_xticks 写成"100 米 / 1 公里 / 30 公里"—— 对数轴要标**实际数值**，不写 10²、10³
#   · clip(lower=10) —— 对数轴上 0 画不出来，把 0 米并到 10 米这一档
# 标题里的数字：p500 = 最近距离 ≤ 500 米的人占比，和图上那个标注点是同一个变量。
p500 = (user_min <= 500).mean()
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.ecdf(user_min.clip(lower=10), color=MAIN, lw=2.2)
ax.set_xscale("log")
ax.axvline(500, color=MUTED, lw=1, ls=(0, (4, 3)))
ax.annotate(f"500 米以内：{p500:.0%}", (500, p500), xytext=(10, -18), textcoords="offset points",
            fontsize=10.5, color=INK, fontweight="bold")
ax.scatter([500], [p500], color=MAIN, zorder=3)
ax.set_xticks([100, 300, 1000, 3000, 10_000, 30_000], ["100 米", "300 米", "1 公里", "3 公里", "10 公里", "30 公里"])
ax.yaxis.set_major_formatter("{x:.0%}")
ax.set_xlabel("当天离西塔最近的距离（对数刻度）", color=MUTED)
ax.grid(axis="x", visible=False)
title = f"{p500:.0%} 的用户当天到过西塔 500 米以内"
viz.finish(fig, title,
           "读法：曲线上一点 = 当天离西塔最近距离不超过横轴数值的用户占比；距离按所连站点的坐标计算；口径：dwd_signal × dim_cell",
           SOURCE, "D4_03_离西塔最近距离.png")

In [ ]:
# 对账
assert 0.84 < p500 < 0.88
print(f"✅ 对账通过：{p500:.1%} 的用户到过西塔 500 米以内")

In [ ]:
from signal_mysql import database as mysql
# 这一格在做什么：算两条曲线——每小时有多少人在西塔，和每个人"第一次"出现在西塔的时刻。
# 两条曲线回答的是不同的问题：前者是**存量**（此刻有多少人在），后者是**流量**（此刻来了多少新人）。
# 讲台提示：D1 留的悬念（23 点信令最多）就靠这一格揭晓。
near = sig[sig["到西塔(米)"] <= 500]
# reindex(range(24)) 补齐 24 个小时：某小时一个人都没有时 groupby 会直接少一行，
# 折线图上看不出断档（D1.7 讲过同一个坑）
hourly_near = near.groupby(near["ts"].dt.hour)["imsi"].nunique().reindex(range(24), fill_value=0)
first_hour = near.groupby("imsi")["ts"].min().dt.hour.value_counts(normalize=True).reindex(range(24), fill_value=0)
xita_hourly = pd.DataFrame({"hour": range(24), "n_present": hourly_near.values,
                            "first_seen_share": first_hour.values})
mysql.write_frame(xita_hourly, "ads_xita_hourly", con, if_exists="replace", index=False)
xita_hourly.iloc[14:21]

In [ ]:
from signal_mysql import database as mysql
x = mysql.read_frame("SELECT * FROM ads_xita_hourly", con)
peak = x.loc[x["n_present"].idxmax()]
day_avg = x.loc[x["hour"].between(9, 16), "n_present"].mean()
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.axvspan(16.5, 23.5, color=MAIN_LIGHT, lw=0, zorder=0)
ax.plot(x["hour"], x["n_present"], color=MAIN, lw=2.4, marker="o", ms=4)
ax.axhline(day_avg, color=MUTED, lw=1, ls=(0, (4, 3)))
ax.text(9, day_avg - 25, f"9–16 点平均 {day_avg:,.0f} 人", va="top", fontsize=9.5, color=MUTED)
ax.annotate(f"{int(peak['hour'])} 点 {int(peak['n_present']):,} 人", (peak["hour"], peak["n_present"]), xytext=(0, 10),
            textcoords="offset points", ha="center", fontsize=10.5, color=INK, fontweight="bold")
ax.set_xticks(range(0, 24, 3), [f"{h} 点" for h in range(0, 24, 3)])
ax.set_xlim(-0.5, 23.5)
ax.set_ylim(0, peak["n_present"] * 1.15)
ax.yaxis.set_major_formatter("{x:,.0f}")
title = f"西塔在场人数 17 点后猛增，{int(peak['hour'])} 点达到 {int(peak['n_present']):,} 人"
viz.finish(fig, title,
           "2018-10-03 每小时出现在西塔 500 米以内的去重用户数；浅色底 = 17 点以后；口径：dwd_signal × dim_cell",
           SOURCE, "D4_04_西塔每小时在场人数.png")

In [ ]:
# ── 出图 D4_05：上下两个面板 ──────────────────────────────────────────────
# 为什么**不用双 Y 轴**：上面是"人"（几千），下面是"占比"（0–40%），单位和量级都不同。
# 双轴画出来，两条线的交点、谁高谁低完全由刻度决定，换个刻度就能得出相反结论。
# 规范的做法是 sharex=True 的上下两个面板，各自的纵轴都从 0 开始。
# （副标题里也明写了"上下两个面板共用横轴（不用双轴）"，让读者知道这是刻意选择。）
# 标题里的两个数字：p17_19（17–19 点首次出现的人数占比之和）、pk（在场人数最高的那一行）。
pk = x.loc[x["n_present"].idxmax()]
p17_19 = x.loc[x["hour"].between(17, 19), "first_seen_share"].sum()
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 5.6), layout="constrained", sharex=True, height_ratios=[1, 1])
for ax_ in (a1, a2):
    ax_.axvspan(16.5, 19.5, color=MAIN_LIGHT, lw=0, zorder=0)
a1.plot(x["hour"], x["n_present"], color=MAIN, lw=2.2, marker="o", ms=3.5)
a1.text(pk["hour"] - 0.5, pk["n_present"] * 1.07, f"{int(pk['hour'])} 点 {int(pk['n_present']):,} 人", ha="right", va="center",
        fontsize=9.5, color=INK, fontweight="bold")
a1.set_ylim(0, x["n_present"].max() * 1.15)
a1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
a1.set_title("西塔 500 米内每小时在场人数（人）", loc="left", fontsize=10, color=INK, fontweight="bold")
a2.plot(x["hour"], x["first_seen_share"], color=GRAY_DARK, lw=2, marker="o", ms=3.5)
e_ = x[x["hour"].between(17, 19)]
a2.plot(e_["hour"], e_["first_seen_share"], color=MAIN, lw=2.6, marker="o", ms=5)
a2.text(18, e_["first_seen_share"].max() + 0.015, f"17–19 点合计 {p17_19:.0%}", ha="center", fontsize=9.5, color=MAIN, fontweight="bold")
a2.text(0.3, x["first_seen_share"].iloc[0] + 0.012, f"0 点已在西塔 {x['first_seen_share'].iloc[0]:.0%}（前一晚没走）", fontsize=9, color=MUTED)
a2.set_ylim(0, x["first_seen_share"].max() * 1.3)
a2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
a2.set_title("当天第一次出现在西塔的时刻（占到过西塔的人）", loc="left", fontsize=10, color=INK, fontweight="bold")
a2.set_xticks(range(0, 24, 3), [f"{h} 点" for h in range(0, 24, 3)]); a2.set_xlim(-0.5, 23.5)
title = f"{p17_19:.0%} 的人第一次出现在西塔是 17–19 点，在场人数随之在 {int(pk['hour'])} 点升到 {int(pk['n_present']):,}"
viz.finish(fig, title,
           "到过西塔 500 米内的用户；上下两个面板共用横轴（不用双轴）；浅蓝底 = 17–19 点；口径：dwd_signal × dim_cell",
           SOURCE, "D4_05_首次出现时间.png")

In [ ]:
# 对账
assert peak["hour"] in (19, 20) and 0.35 < p17_19 < 0.45
print(f"✅ 对账通过：{int(peak['hour'])} 点在场人数最多；{p17_19:.0%} 的人首次出现在 17–19 点")

In [ ]:
# D1 的疑点：为什么 23 点信令最多？看看各小时的信号有多少发生在西塔 1 公里内
share_near = sig.assign(hour=sig["ts"].dt.hour, near_xita=sig["到西塔(米)"] <= 1000).groupby("hour")["near_xita"].mean()
print(f"10 点的信令 {share_near[10]:.0%} 发生在西塔 1 公里内；23 点是 {share_near[23]:.0%}")

In [ ]:
# 这一格在做什么：从 IMSI 里读运营商代码，看样本覆盖了哪几家运营商。
# IMSI 的结构是「3 位国家码 + 2 位运营商码 + 用户编号」，所以 str[3:5] 就是运营商。
# 结果全是中国移动——这意味着**人数只代表移动用户**，要推全市总人数必须按移动的市场份额扩样。
# 这条限制和 D4.5 的样本圈选一样，都要写进"不能回答什么"。
mnc = sig.drop_duplicates("imsi")["imsi"].str[3:5].value_counts().rename_axis("运营商代码").reset_index(name="n_users")
mnc["含义"] = mnc["运营商代码"].map({"00": "中国移动", "02": "中国移动", "07": "中国移动"}).fillna("其他")
mnc

In [ ]:
quality.dq_record(con, "DQ-15", "代表性", "样本圈选口径", "dwd_signal",
          f"{top['visitor_share']:.0%} 的用户到过和平区，{p500:.0%} 到过西塔 500 米内", "警告",
          evidence="热点位于西塔；在场人数 17 点后猛增；首次出现时间 16→17 点跳升过陡",
          decision="结论只针对“10 月 3 日傍晚后到访西塔的人群”；圈选条件需向数据提供方确认")
quality.dq_record(con, "DQ-16", "代表性", "运营商覆盖", "dwd_signal.imsi", "全部为中国移动用户", "警告",
          evidence="IMSI 运营商代码只有 00、02、07", decision="人数只代表移动用户；推算总人数需按移动市场份额扩样")

In [ ]:
# 这一格在做什么：把"每人每小时在哪"算出来，再归到五类区域里。
# 为什么取**中位数**而不是平均：一小时内的几条信令可能分散在好几个站点，
# 平均会落到几个站点中间的空地上（甚至落到河里）；中位数一定落在实际出现过的位置附近。
# 讲台提示：这是一个粗糙版本，D9 会用停留点重做一遍 dws_user_hour，数字会略有变化——
# 副标题里已经先说明了这件事，不要让学生以为两处数字对不上是算错了。
uh = sig.assign(hour=sig["ts"].dt.hour).groupby(["imsi", "hour"], as_index=False)[["lng", "lat"]].median()
uh["district"] = pd.Series(geo.point_in_polygons(uh["lng"], uh["lat"], DISTRICTS)).fillna("市外").to_numpy()
uh["aoi"] = geo.point_in_polygons(uh["lng"], uh["lat"], AOI)
uh["dist_xita_m"] = geo.haversine(uh["lat"], uh["lng"], hot_lat, hot_lng)
CITY = ["沈河区", "皇姑区", "铁西区", "大东区", "浑南区", "于洪区", "沈北新区", "苏家屯区"]
order = ["西塔 500 米内", "太原街、中街商圈", "和平区其他", "其他城区", "远郊县市及市外"]
uh["area"] = np.select([uh["dist_xita_m"] <= 500, uh["aoi"].isin(["太原街商圈", "中街商圈"]),
                        uh["district"] == "和平区", uh["district"].isin(CITY)], order[:4], default=order[4])
cnt = uh.groupby(["hour", "area"]).size().unstack(fill_value=0).reindex(index=range(24), columns=order, fill_value=0)
share = cnt.div(cnt.sum(axis=1), axis=0) * 100
share.round(1).loc[[4, 10, 15, 19, 23]]

In [ ]:
# 对账
day_out = (share.loc[10:16, "其他城区"] + share.loc[10:16, "远郊县市及市外"]).mean()
xs = share["西塔 500 米内"]
assert len(uh) == 86_563 and round(day_out, 1) == 65.0 and int(xs.idxmax()) == 19
print(f"✅ 对账通过：{len(uh):,} 个“人 × 小时”；白天 10–16 点 {day_out:.1f}% 在和平区以外；西塔占比 19 点最高（{xs.max():.1f}%）")

In [ ]:
# ── 出图 D4_06：堆叠面积图 ───────────────────────────────────────────────
# 为什么这次可以堆叠：这五类区域是**互斥且穷尽**的（每人每小时只落进一类，合计正好 100%），
# 所以堆起来有意义。D4_01 那张"各区到访占比"就不能堆——一个人能到过多个区，合计超过 100%。
# 判断标准就一条：**加起来等于 100% 才能堆叠**。
# 规范落在哪几行：
#   · 类别压到 5 类 —— 堆叠超过 5 层，中间几层的厚度就读不出来了
#   · 右侧 annotate 直接标区域名 —— 不用图例（图例要来回对照颜色）
#   · ys[i] = max(ys[i], ys[i-1] + 7) —— 标签挤在一起时往上推开，避免文字重叠
#     （run_notebooks.py 会自动检查文字重叠，这里是提前规避）
# 标题里的数字：day_out（10–16 点在和平区以外的平均占比）、xs[peak_h]（西塔占比的峰值）。
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.stackplot(share.index, share.T.values, colors=[MAIN, "#7FB0D6", "#E4E7EB", GRAY, "#98A2AE"],
             edgecolor="white", linewidth=0.8)
ax.set_xlim(0, 23)
ax.set_ylim(0, 100)
ax.set_xticks(range(0, 24, 3), [f"{h} 点" for h in range(0, 24, 3)])
ax.set_yticks(range(0, 101, 25), [f"{v}%" for v in range(0, 101, 25)])
ax.grid(False)
# 在右边直接标区域名：标签按 23 点各层的中点摆放，挤在一起的往上推开
mids = (share.loc[23].cumsum() - share.loc[23] / 2).to_numpy()
ys = mids.copy()
for i in range(1, len(ys)):
    ys[i] = max(ys[i], ys[i - 1] + 7)
for g, ym, y in zip(order, mids, ys):
    ax.annotate(g, xy=(23, ym), xytext=(23.6, min(y, 97)), va="center", fontsize=10, annotation_clip=False,
                color=MAIN if g.startswith("西塔") else INK, fontweight="bold" if g.startswith("西塔") else "normal",
                arrowprops=dict(arrowstyle="-", color=FAINT, lw=0.6, shrinkA=0, shrinkB=0))
peak_h = int(xs.idxmax())
ax.text(peak_h, xs[peak_h] / 2, f"{peak_h} 点\n{xs[peak_h]:.0f}%", ha="center", va="center", color="white", fontsize=10, fontweight="bold")
c12 = share.loc[12].cumsum()
ax.text(12, c12["和平区其他"] + share.loc[12, "其他城区"] / 2, f"白天 {day_out:.0f}% 在和平区以外", ha="center", va="center",
        color=INK, fontsize=11)
title = f"白天 {day_out:.0f}% 的人在和平区以外，{peak_h} 点有 {xs[peak_h]:.0f}% 聚到西塔：样本圈的是人，不是地方"
viz.finish(fig, title,
           "每小时有信令的样本用户在各区域的占比；位置 = 该小时信令位置的中位数；西塔 = 热点 500 米内，商圈 = v2 边界；\n"
           "D9 会用停留点修正位置，数字略有变化；口径：dwd_signal × dim_cell",
           SOURCE, "D4_06_城市潮汐.png")

In [ ]:
import re
def localize_html(html, local_dir="local"):
    """把 HTML 里引用的在线 JS/CSS 地址改成相对路径：local/js/文件名、local/css/文件名。"""
    html = re.sub(r'src="https?://[^"]+/([^/"]+\.js)"', rf'src="{local_dir}/js/\1"', html)
    html = re.sub(r'href="https?://[^"]+/([^/"]+\.css)"', rf'href="{local_dir}/css/\1"', html)
    return html
sample = ('<script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>\n'
          '<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>')
print(localize_html(sample))

In [ ]:
con.close()